# Checkpoint Evaluation and Run Comparison

Use this notebook to compare experiment runs and optionally re-evaluate saved checkpoints.

Customize in Cell 3:
- `RUN_DIR_PATTERNS`
- `CHECKPOINTS_TO_REEVALUATE`
- `METRICS_TO_PLOT` and `TASK_METRIC_TO_PLOT`

In [ ]:
from pathlib import Path
import glob
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

print(f'Repository root: {REPO_ROOT}')

In [ ]:
RESULTS_DIR = REPO_ROOT / 'results'
RUN_DIR_PATTERNS = ['vit_cms_hybrid_cad_*']
CHECKPOINTS_TO_REEVALUATE = []
MAX_TASKS_FOR_REEVAL = 2
METRICS_TO_PLOT = ['avg_eval_accuracy', 'avg_eval_f1', 'avg_eval_auroc']
TASK_METRIC_TO_PLOT = 'f1'

RESULTS_DIR

In [ ]:
def find_run_dirs(results_dir: Path, patterns):
    run_dirs = []
    for pattern in patterns:
        run_dirs.extend(Path(p) for p in glob.glob(str(results_dir / pattern)))
    run_dirs = [p for p in run_dirs if p.is_dir()]
    run_dirs.sort(key=lambda p: p.name)
    return run_dirs


def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_run_summary(run_dir: Path):
    summary_path = run_dir / 'run_summary.json'
    if not summary_path.exists():
        return None
    summary = load_json(summary_path)
    summary['run_dir_name'] = run_dir.name
    summary['run_dir_path'] = str(run_dir)
    return summary


def load_task_table(run_dir: Path):
    metrics_paths = sorted(run_dir.glob('task_*_metrics.json'))
    rows = []
    for path in metrics_paths:
        data = load_json(path)
        eval_metrics = data.get('eval', {})
        row = {
            'run_dir_name': run_dir.name,
            'task_id': data.get('task_id'),
            'category': data.get('category'),
        }
        row.update(eval_metrics)
        rows.append(row)
    return pd.DataFrame(rows)


run_dirs = find_run_dirs(RESULTS_DIR, RUN_DIR_PATTERNS)
summaries = [load_run_summary(run_dir) for run_dir in run_dirs]
summaries = [s for s in summaries if s is not None]
summary_df = pd.DataFrame(summaries)

task_tables = [load_task_table(run_dir) for run_dir in run_dirs]
task_df = pd.concat(task_tables, ignore_index=True) if task_tables else pd.DataFrame()

print(f'Found {len(run_dirs)} run folders')
if not summary_df.empty:
    display(summary_df[['run_dir_name', 'tasks_completed'] + [m for m in METRICS_TO_PLOT if m in summary_df.columns]])
if not task_df.empty and TASK_METRIC_TO_PLOT in task_df.columns:
    display(task_df[['run_dir_name', 'task_id', 'category', TASK_METRIC_TO_PLOT]].head(20))

In [ ]:
if summary_df.empty:
    print('No run summaries found with current patterns.')
else:
    plot_df = summary_df[['run_dir_name'] + [m for m in METRICS_TO_PLOT if m in summary_df.columns]].copy()
    long_df = plot_df.melt(id_vars='run_dir_name', var_name='metric', value_name='value')

    plt.figure(figsize=(12, 5))
    sns.barplot(data=long_df, x='run_dir_name', y='value', hue='metric')
    plt.xticks(rotation=25, ha='right')
    plt.title('Run-level Metric Comparison')
    plt.xlabel('Run')
    plt.ylabel('Value')
    plt.tight_layout()
    plt.show()

if task_df.empty or TASK_METRIC_TO_PLOT not in task_df.columns:
    print('Task-level table is empty or metric is not available.')
else:
    pivot_df = task_df.pivot_table(index='task_id', columns='run_dir_name', values=TASK_METRIC_TO_PLOT, aggfunc='mean')
    plt.figure(figsize=(12, 4 + 0.4 * len(pivot_df.index)))
    sns.heatmap(pivot_df, annot=True, fmt='.2f', cmap='YlGnBu')
    plt.title(f'Task-level {TASK_METRIC_TO_PLOT} Comparison')
    plt.xlabel('Run')
    plt.ylabel('Task ID')
    plt.tight_layout()
    plt.show()

In [ ]:
import torch

from dataset.load_dataset import ContinualStreamingManager
from training.evaluator import Evaluator
from training.run_experiment import build_model


def reevaluate_checkpoint(checkpoint_path: Path, max_tasks=2, device='auto'):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    config = checkpoint['config']

    model = build_model(config)
    model.load_state_dict(checkpoint['model_state_dict'])

    if device == 'auto':
        chosen_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    else:
        chosen_device = device

    evaluator = Evaluator(model=model, device=chosen_device, task_type=config.get('training', {}).get('task_type', 'anomaly'))
    manager = ContinualStreamingManager(config)

    rows = []
    task_count = 0
    while True:
        _, test_loader, info = manager.get_next_task()
        if test_loader is None:
            break

        metrics = evaluator.evaluate_task(test_loader, task_id=info['task_id'], verbose=False)
        rows.append({'checkpoint': checkpoint_path.name, 'task_id': info['task_id'], 'category': info['category'], **metrics})

        task_count += 1
        if max_tasks is not None and task_count >= max_tasks:
            break

    return pd.DataFrame(rows)


reeval_tables = []
for ckpt in CHECKPOINTS_TO_REEVALUATE:
    ckpt_path = Path(ckpt)
    if not ckpt_path.is_absolute():
        ckpt_path = REPO_ROOT / ckpt_path
    if ckpt_path.exists():
        reeval_tables.append(reevaluate_checkpoint(ckpt_path, max_tasks=MAX_TASKS_FOR_REEVAL))

reeval_df = pd.concat(reeval_tables, ignore_index=True) if reeval_tables else pd.DataFrame()
display(reeval_df.head(20))